# AdCanvas 메인 노트북 (배포용)

사진 한 장 → **① 누끼** → ② VLM(요청 해석) → ③ 배경 생성 → ④ 원본 음식 재합성

* 각 단계는 앞 단계 결과를 정해진 형식(`handoff`)으로 받아서 씀
* 평가(점수, 비교 실험)는 서브 노트북에서 함. 이 노트북은 **실행만**
* 런타임: GPU(T4) 권장. 허깅페이스 토큰 필요 없음

# 1. 누끼 (Segmentation)

**사진 → 그릇 후보 → (사용자 선택) → 음식 마스크 + 투명 배경 누끼**

| 단계 | 모델 | 역할 |
|---|---|---|
| 그릇 찾기 | Grounding DINO tiny | 글자(plate, bowl …)로 그릇·음식 박스 찾기 |
| 그릇별 마스크 | SAM 2.1 hiera-small + **우리 파인튜닝 디코더** | 박스 하나 → 그릇 + 음식 + 걸친 수저를 통째로 |
| 후처리 | 직접 작성 | 조각 정리, 구멍 채우기, 중복 그릇 제거 |

* 파인튜닝 가중치: 드라이브 `AdCanvas_nukki/weights/sam2_ft.pt` (없으면 원래 SAM으로 동작)
* 결과 마스크: **흰색(255) = 음식(그릇 포함)**, 원본 사진과 같은 크기

## 1-0. 설치

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif

## 1-1. 설정
* `NUKKI_WEIGHTS`: 파인튜닝 가중치 위치 (팀 드라이브 공유 폴더 기준)
* `OUT_DIR`: 결과 저장 위치 (다음 단계가 여기서 읽음)

In [ ]:
import os, io, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from google.colab import drive
drive.mount('/content/drive')

NUKKI_BASE = '/content/drive/MyDrive/AdCanvas_nukki'
NUKKI_WEIGHTS = f'{NUKKI_BASE}/weights/sam2_ft.pt'
OUT_DIR = '/content/adcanvas_out'
os.makedirs(OUT_DIR, exist_ok=True)
print('파인튜닝 가중치:', '있음' if os.path.exists(NUKKI_WEIGHTS) else '없음 (원래 SAM 사용)')

## 1-2. 누끼 함수
실행만 하면 됨. 바깥에서 쓰는 함수:

| 함수 | 입력 → 출력 |
|---|---|
| `load()` | 모델 불러오기 (한 번) |
| `candidates(img)` | 사진 → 그릇 후보 목록 `[{'id','label','score','kind','dish','box','mask'}]` |
| `merge(img, cands, ids)` | 고른 후보만 합친 마스크 (ids=None이면 기본 선택 전부) |
| `segment(img)` | 사진 → 마스크 (자동, 기본 선택 전부) |
| `refine(img, box, points, labels)` | 사용자 박스·클릭 보정 → 그릇 하나 마스크 |
| `cutout(img, mask)` | 투명 배경 누끼 (RGBA) |

In [ ]:
import os

import cv2
import numpy as np
import torch
from PIL import Image, ImageOps
from scipy import ndimage

SAM_MODEL = 'facebook/sam2.1-hiera-small'
GD_ID = 'IDEA-Research/grounding-dino-tiny'
MAX_SIDE = 2048            # 이보다 큰 사진은 줄여서 처리하고, 마스크는 원래 크기로 되돌림

# 1_pipeline.ipynb와 같은 값 (11/16 test IoU 0.906, BF 0.827)
DINO_CFG = dict(
    dish_words=['plate', 'bowl', 'dish', 'pot'],            # 그릇 (기본 선택)
    other_words=['cup', 'glass', 'spoon', 'chopsticks'],    # 후보로만 (기본 선택 해제)
    food_words=['food'],                                    # 그릇 없이 놓인 음식
    box_th=0.3, nms_iou=0.6, max_area=0.9, food_max_area=0.5,
)
CAND = dict(
    utensil_words=['spoon', 'chopsticks'],
    utensil_overlap=0.2,   # 수저의 20% 이상이 그릇 외곽 안에 있으면 걸친 것 (원래 SAM일 때만 합침)
    part_min=0.1,          # 마스크 조각 중 가장 큰 조각의 10% 이상만 남김
    solidity=0.9,          # 원래 SAM일 때: 그릇이 볼록 외곽의 90% 미만이면 외곽으로 채움
    inside_drop=0.9,       # 다른 그릇 안에 90% 이상 들어간 후보는 중복으로 제거
    sam_chunk=4,           # SAM을 박스 몇 개씩 돌릴지 (메모리 절약)
)

_m = {}   # 불러온 모델: device, gd_proc, gd, sam, finetuned


# ---------------- 불러오기 ----------------
def _device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


def load(weights=NUKKI_WEIGHTS):
    """모델 불러오기 (서버 켤 때 한 번). 반환: 파인튜닝 가중치를 썼는지"""
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    try:
        from pillow_heif import register_heif_opener
        register_heif_opener()   # 아이폰 HEIC
    except ImportError:
        pass
    dev = _device()
    _m['device'] = dev
    _m['gd_proc'] = AutoProcessor.from_pretrained(GD_ID)
    _m['gd'] = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(dev).eval()
    sam = SAM2ImagePredictor.from_pretrained(SAM_MODEL, device=dev)
    _m['finetuned'] = bool(weights) and os.path.exists(weights)
    if _m['finetuned']:
        res = sam.model.load_state_dict(torch.load(weights, map_location=dev), strict=False)
        assert not res.missing_keys and not res.unexpected_keys, 'sam2_ft.pt와 SAM_MODEL이 맞지 않음'
    _m['sam'] = sam
    _m['sam_img'] = None   # set_image 한 사진 (클릭 보정 때 다시 계산하지 않으려고)
    return _m['finetuned']


def _ready():
    if 'sam' not in _m:
        load()


def _prep(img):
    """PIL 사진 → (처리용 RGB 사진, 원래 크기). EXIF 회전 반영, 긴 변 MAX_SIDE로"""
    img = ImageOps.exif_transpose(img).convert('RGB')
    size = img.size
    if max(size) > MAX_SIDE:
        img = img.copy()
        img.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
    return img, size


def _to_size(m, size):
    """bool 마스크 → 원래 크기의 흑백 PIL (L, 0/255)"""
    out = Image.fromarray((m * 255).astype(np.uint8))
    return out if out.size == size else out.resize(size, Image.NEAREST)


def _set_image(img_np):
    key = (img_np.shape, int(img_np[::97, ::89].sum()))
    if _m['sam_img'] != key:
        _m['sam'].set_image(img_np)
        _m['sam_img'] = key


# ---------------- DINO ----------------
def _box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


@torch.no_grad()
def dino_boxes(img):
    """사진 → [{'box', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택)}]"""
    from torchvision.ops import nms
    C, proc, gd = DINO_CFG, _m['gd_proc'], _m['gd']
    words = C['dish_words'] + C['other_words'] + C['food_words']
    kinds = ['dish'] * len(C['dish_words']) + ['other'] * len(C['other_words']) + ['food'] * len(C['food_words'])
    inputs = proc(images=img, text='. '.join(words) + '.', return_tensors='pt').to(_m['device'])
    out = gd(**inputs)
    probs = out.logits[0].sigmoid().float().cpu()
    ids = inputs['input_ids'][0].tolist()
    cols = []
    for w in words:
        sub = proc.tokenizer(w, add_special_tokens=False)['input_ids']
        pos = [i + j for i in range(len(ids) - len(sub) + 1) if ids[i:i + len(sub)] == sub for j in range(len(sub))]
        cols.append(probs[:, pos].max(1).values if pos else torch.zeros(len(probs)))
    score, best = torch.stack(cols, 1).max(1)
    W, H = img.size
    cx, cy, bw, bh = out.pred_boxes[0].float().cpu().unbind(-1)
    boxes = torch.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = [i for i in range(len(boxes)) if score[i] > C['box_th'] and _box_area(boxes[i].tolist()) <
            (C['food_max_area'] if kinds[int(best[i])] == 'food' else C['max_area']) * W * H]
    if not keep:
        return []
    keep = torch.tensor(keep)
    k = keep[nms(boxes[keep], score[keep], C['nms_iou'])].tolist()
    return [{'box': boxes[i].tolist(), 'label': words[int(best[i])], 'score': float(score[i]),
             'kind': kinds[int(best[i])], 'dish': kinds[int(best[i])] != 'other'} for i in k]


# ---------------- 마스크 도우미 ----------------
def _box_mask(shape, box, expand=0.1):
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def _local(fn, m, pad=0):
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def _hull(m):
    def f(c):
        out = np.zeros(c.shape, np.uint8)
        cv2.fillConvexPoly(out, cv2.convexHull(cv2.findNonZero(c.astype(np.uint8))), 1)
        return out.astype(bool)
    return _local(f, m)


def _close_fill(m):
    return _local(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m, pad=4)


def _keep_parts(c, box):
    lab, n = ndimage.label(c)
    if n <= 1:
        return c
    idx = np.arange(1, n + 1)
    sizes = ndimage.sum(c, lab, idx)
    inside = ndimage.sum(_box_mask(c.shape, box), lab, idx)
    keep = idx[(sizes >= CAND['part_min'] * sizes.max()) & (inside >= 0.5 * sizes)]
    return np.isin(lab, keep if len(keep) else idx[[sizes.argmax()]])


def _crop(m, box, fn, pad=4):
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return m
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out = np.zeros_like(m)
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1], [box[0] - x0, box[1] - y0, box[2] - x0, box[3] - y0])
    return out


def _pick_whole(ms, ss, box, mm_score=0.6, mm_inside=0.9):
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= mm_score * ss.max()]
    bm = _box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= mm_inside * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


def _clean_dish(m, box):
    out = _crop(m, box, lambda c, b: ndimage.binary_fill_holes(ndimage.binary_closing(_keep_parts(c, b), iterations=3)))
    if not out.any():
        return out
    h = _hull(out) & (_box_mask(m.shape, box, 0.05) | out)
    return h if out.sum() < CAND['solidity'] * h.sum() else out


# ---------------- SAM + 후처리 ----------------
@torch.no_grad()
def _sam_masks(img_np, boxes, kinds):
    """파인튜닝 SAM은 마스크 1개, 원래 SAM은 3개 중 전체 물체를 고름"""
    _set_image(img_np)
    H, W = img_np.shape[:2]
    multi = not _m['finetuned']
    out = []
    for s0 in range(0, len(boxes), CAND['sam_chunk']):
        b = np.array(boxes[s0:s0 + CAND['sam_chunk']], np.float32)
        m, sc, _ = _m['sam'].predict(box=b, multimask_output=multi)
        m = np.asarray(m).reshape(len(b), -1, H, W) > 0.5
        if not multi:
            out += list(m[:, 0])
            continue
        sc = np.asarray(sc).reshape(len(b), -1)
        for k in range(len(b)):
            out.append(m[k, int(sc[k].argmax())] if kinds[s0 + k] == 'other' else _pick_whole(m[k], sc[k], boxes[s0 + k]))
    return out


def _tidy(cands, light):
    for c in cands:
        if c['dish'] and c['kind'] != 'food':
            c['mask'] = _close_fill(_crop(c['mask'], c['box'], _keep_parts, pad=0)) if light else _clean_dish(c['mask'], c['box'])
    dishes = sorted([c for c in cands if c['dish'] and c['kind'] != 'food'], key=lambda c: -c['mask'].sum())
    for i, small in enumerate(dishes):
        for big in dishes[:i]:
            if 'dup_of' not in big and (small['mask'] & big['mask']).sum() >= CAND['inside_drop'] * max(small['mask'].sum(), 1):
                small['dish'], small['dup_of'] = False, big['id']
                break
    kept = [d for d in dishes if 'dup_of' not in d]
    union = np.zeros(cands[0]['mask'].shape, bool)
    for d in kept:
        union |= d['mask']
    for f in [c for c in cands if c['kind'] == 'food']:
        f['mask'] = _close_fill(_crop(f['mask'], f['box'], _keep_parts, pad=0))
        a = f['mask'].sum()
        swallows = any((f['mask'] & d['mask']).sum() > 0.3 * d['mask'].sum() for d in kept)
        if a == 0 or (f['mask'] & union).sum() >= 0.5 * a or swallows:
            f['dish'], f['dup_of'] = False, -1
    utensils = [u for u in cands if u['label'] in CAND['utensil_words'] and u['mask'].any()]
    if utensils and not light:
        hulls = {c['id']: _hull(c['mask']) for c in cands if c['dish'] and 'dup_of' not in c and c['mask'].any()}
        for u in utensils:
            if not hulls:
                break
            best = max(hulls, key=lambda i: (u['mask'] & hulls[i]).sum())
            if (u['mask'] & hulls[best]).sum() > CAND['utensil_overlap'] * u['mask'].sum():
                d = next(c for c in cands if c['id'] == best)
                d['mask'] = _close_fill(d['mask'] | u['mask'])
                u['merged_into'] = best
    return [c for c in cands if 'dup_of' not in c]


# ---------------- 바깥에서 쓰는 함수 ----------------
def candidates(img):
    """사진 → 그릇 후보 목록. 박스는 원래 사진 좌표, mask는 원래 크기의 흑백 PIL
    [{'id', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택), 'box'[x0,y0,x1,y1], 'mask'}]
    그릇을 하나도 못 찾으면 [] (서비스에서는 사용자가 직접 박스를 그리게 → refine)"""
    _ready()
    work, size = _prep(img)
    found = dino_boxes(work)
    if not any(f['dish'] for f in found):
        return []
    masks = _sam_masks(np.array(work), [f['box'] for f in found], [f['kind'] for f in found])
    for i, (f, m) in enumerate(zip(found, masks)):
        f.update(id=i, mask=m)
    cands = _tidy(found, light=_m['finetuned'])
    s = size[0] / work.size[0]
    return [{'id': c['id'], 'label': c['label'], 'score': round(c['score'], 3), 'kind': c['kind'], 'dish': c['dish'],
             'box': [round(v * s, 1) for v in c['box']], 'mask': _to_size(c['mask'], size)} for c in cands]


def merge(img, cands, ids=None):
    """후보 중 고른 것만 합친 마스크 (ids=None이면 기본 선택된 그릇 전부)"""
    W, H = ImageOps.exif_transpose(img).size
    out = np.zeros((H, W), bool)
    for c in cands:
        if (c['id'] in ids) if ids is not None else c['dish']:
            out |= np.asarray(c['mask']) > 127
    return _to_size(out, (W, H))


def segment(img):
    """팀 공통 함수 모양 (KAN-5): RGB 사진 → 흑백 마스크 (L, 사진과 같은 크기, 255 = 음식+그릇)"""
    return merge(img, candidates(img))


@torch.no_grad()
def refine(img, box, points=None, labels=None):
    """클릭 보정: 박스(원래 사진 좌표) + 클릭 점(1 포함, 0 제외) → 그 그릇 하나의 마스크 (L, 원래 크기)"""
    _ready()
    work, size = _prep(img)
    s = work.size[0] / size[0]
    _set_image(np.array(work))
    kw = dict(box=np.array([v * s for v in box], np.float32), multimask_output=False)
    if points:
        kw.update(point_coords=np.array(points, np.float32) * s, point_labels=np.array(labels, np.int32))
    m, _, _ = _m['sam'].predict(**kw)
    return _to_size(np.asarray(m)[0] > 0.5, size)


def cutout(img, mask, crop=False):
    """원본 + 마스크 → 투명 배경 누끼 (RGBA). crop=True면 음식 부분만 잘라냄"""
    rgb = ImageOps.exif_transpose(img).convert('RGB')
    out = rgb.copy()
    out.putalpha(mask.convert('L').resize(rgb.size, Image.NEAREST))
    return out.crop(out.getbbox()) if crop and out.getbbox() else out

## 1-3. 모델 불러오기

In [ ]:
t0 = time.time()
ft = load(NUKKI_WEIGHTS)
print(f"device: {_m['device']} | 파인튜닝 SAM: {ft} | {time.time() - t0:.1f}초")

## 1-4. 사진 넣기
* `IMG_PATH`에 경로를 넣거나, 비워 두면 업로드 창이 뜸

In [ ]:
IMG_PATH = ''   # 예: f'{NUKKI_BASE}/dataset/images/hwang/IMG_0405.HEIC'

if not IMG_PATH:
    from google.colab import files
    up = files.upload()
    IMG_PATH = f'/content/{next(iter(up))}'
img = ImageOps.exif_transpose(Image.open(IMG_PATH)).convert('RGB')
print(IMG_PATH, img.size)

## 1-5. 그릇 후보 찾기
* 초록 = 그릇(기본 선택), 주황 = 컵·수저 등(기본 선택 안 함)
* 그릇을 하나도 못 찾으면 1-7(직접 박스)로

In [ ]:
COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128)])


def show_candidates(img, cands):
    base = np.array(img)
    over = base.copy()
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(base)
    for k, c in enumerate(cands):
        x0, y0, x1, y1 = c['box']
        col = 'lime' if c['dish'] else 'orange'
        ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=col, lw=2))
        ax[0].text(x0, y0, f"{c['id']}:{c['label']} {c['score']:.2f}", fontsize=8, backgroundcolor=col)
        if c['dish']:
            m = np.asarray(c['mask']) > 127
            over[m] = (over[m] * 0.45 + COLORS[k % len(COLORS)] * 0.55).astype(np.uint8)
    ax[0].set_title('candidates (green = dish)'); ax[1].imshow(over); ax[1].set_title('dish candidates')
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()


t0 = time.time()
cands = candidates(img)
print(f"후보 {len(cands)}개, 그릇 {sum(c['dish'] for c in cands)}개 | {time.time() - t0:.1f}초")
print('그릇 id:', [c['id'] for c in cands if c['dish']])
if cands:
    show_candidates(img, cands)
else:
    print('그릇을 찾지 못했어요. 1-7에서 박스를 직접 지정하세요.')

## 1-6. 그릇 고르기 → 마스크, 누끼
* `SELECT = None`: 기본 선택(그릇 전부) / `SELECT = [0, 2]`: 고른 id만

In [ ]:
SELECT = None

mask = merge(img, cands, SELECT) if cands else None

## 1-7. (선택) 직접 박스·클릭으로 보정
* 자동으로 못 찾았거나 고치고 싶을 때만. `BOX = [x0, y0, x1, y1]` (원본 사진 좌표)
* `POINTS`: 클릭 좌표, `LABELS`: 1 포함 / 0 제외

In [ ]:
BOX = None          # 예: [400, 900, 2600, 3300]
POINTS, LABELS = None, None

if BOX:
    m = refine(img, BOX, POINTS, LABELS)
    mask = m if mask is None else Image.fromarray(np.maximum(np.asarray(mask), np.asarray(m)))
assert mask is not None, '마스크가 없어요. 1-6 또는 1-7을 확인하세요.'

cut = cutout(img, mask)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img); ax[0].set_title('image')
ax[1].imshow(mask, cmap='gray'); ax[1].set_title('mask (white = food)')
ax[2].imshow(cut); ax[2].set_title('cutout')
for a in ax:
    a.axis('off')
plt.tight_layout(); plt.show()

## 1-8. 다음 단계(VLM)에 넘기기
* `OUT_DIR`에 파일로 저장하고, 같은 내용을 `nukki_out` 변수로도 넘김
* **형식 초안** (주말까지 API 코드로 바꿀 예정)

| 키 | 내용 |
|---|---|
| `image` | 원본 사진 경로 (RGB, EXIF 회전 반영) |
| `mask` | 음식 마스크 경로 (L, 흰색 = 음식, 원본과 같은 크기) |
| `cutout` | 투명 배경 누끼 경로 (RGBA, 원본과 같은 크기) |
| `cutout_crop` | 음식 부분만 잘라낸 누끼 경로 |
| `food_box` | 음식 영역 박스 `[x0, y0, x1, y1]` |
| `dishes` | 선택된 그릇 목록 (id, 이름, 점수, 박스) |

In [ ]:
def to_next_stage(img, mask, cands, select, out_dir=OUT_DIR, name='input'):
    """누끼 결과 → 다음 단계가 읽을 형식 (파일 + dict)"""
    paths = {k: f'{out_dir}/{name}_{k}.png' for k in ('image', 'mask', 'cutout', 'cutout_crop')}
    img.save(paths['image'])
    mask.save(paths['mask'])
    cut = cutout(img, mask)
    cut.save(paths['cutout'])
    cutout(img, mask, crop=True).save(paths['cutout_crop'])
    box = cut.getbbox()
    chosen = [c for c in cands if ((c['id'] in select) if select is not None else c['dish'])]
    out = dict(paths, size=list(img.size), food_box=list(box) if box else None,
               dishes=[{k: c[k] for k in ('id', 'label', 'score', 'box')} for c in chosen],
               finetuned=_m['finetuned'])
    json.dump(out, open(f'{out_dir}/{name}_nukki.json', 'w'), ensure_ascii=False, indent=1)
    return out


nukki_out = to_next_stage(img, mask, cands, SELECT)
print(json.dumps({k: v for k, v in nukki_out.items() if k != 'dishes'}, ensure_ascii=False, indent=1))
print('그릇', len(nukki_out['dishes']), '개 선택')

# 2. VLM (요청 해석) — VLM팀
* 입력: `nukki_out` (1-8 형식) + 사용자 요청 문장
* 출력: 생성 프롬프트, 부정 프롬프트, 충돌 여부 → `vlm_out`

In [ ]:
# VLM팀 작성

# 3. 배경 생성 (인페인팅 / 아웃페인팅) — 생성팀
* 입력: `nukki_out['image']`, `nukki_out['mask']`(흰색 = 음식), `vlm_out`
* 출력: 목표 비율의 배경 생성 이미지 → `gen_out`

In [ ]:
# 생성팀 작성

# 4. 원본 음식 재합성 — 생성팀
* 생성 결과 위에 원본 음식 픽셀을 다시 덮어 음식 보존
* 출력: 최종 광고 이미지

In [ ]:
# 생성팀 작성

# 5. 서버 (FastAPI + ngrok) — 주말 작성 예정
* 프론트 `POST /generate` → 1~4 실행 → `{imageUrl, copy}` 응답